## Rotate Developer Credential Keys

#### import modules

In [2]:
from arcgis.gis import GIS
from json import dump, load
from datetime import datetime, timedelta
from pathlib import Path

#### helper functions

In [8]:
def slot_for_key(key:str):
    slot = int(key[-10:-9])
    if slot == 1 or slot == 2:
        return slot

    return None

In [10]:
class cfg:
    item_id = "4f26fde50fda40678d98c575031ee720"
    days_in_future=3
    config_file_path=Path.cwd() / "app-config.json"

#### connect to our portal

In [6]:
# portal = GIS("home")
portal = GIS(profile="DTS2026")
print(f"connected as {portal.properties.user.username} to {portal.properties.name}")

connected as mtorrey_devlabs to ArcGIS DevLabs


#### Read the current API key from our file

In [11]:
app_config_path = cfg.config_file_path
with open(app_config_path,"r") as file:
    data=load(file)

In [12]:
current_api_key = data["apiKey"]

In [13]:
current_api_key_slot = slot_for_key(current_api_key)

In [14]:
print(f"current key is in slot {current_api_key_slot}")

current key is in slot 1


#### Create a new expiration date

In [15]:
new_expiration = (datetime.now()+timedelta(days=3)).replace(hour=23, minute=59, microsecond=999000)
print(f"new expiration date: {new_expiration.strftime("%c")}")

new expiration date: Thu Mar 12 23:59:42 2026


#### get the current developer credential item

In [16]:
developer_credential_item = portal.admin.developer_credentials.get("4f26fde50fda40678d98c575031ee720")
developer_credential_item._item

<Item title:"Scripting Developer Credentials (ADMIN)" type:Application owner:mtorrey_devlabs>

#### create a new token in the open slot

In [17]:
new_slot = 2 if current_api_key_slot == 1 else 1

In [18]:
new_key = developer_credential_item.generate_token(slot=new_slot, expiration=new_expiration)
new_key_token = new_key.get("access_token")

In [19]:
print(f"new key in slot {new_slot} created {new_key_token[-12:]}...")

new key in slot 2 created AT2_mL1xw0pD...


#### write the new key to our app file

In [20]:
data["apiKey"] = new_key_token

In [21]:
print("writing new key to file...")
with open(app_config_path,"w") as f:
    dump(data, f, indent=2)
    print(f"\tnew key written to {app_config_path}")

writing new key to file...
	new key written to /Users/mark4582/_esri/_dev_summit/2026/2026-Scripting-and-Automating-Developer-Credentials/python/app-config.json


#### invalidate the old API Key

In [22]:
developer_credential_item.revoke(current_api_key_slot)

True

In [23]:
print(f"old key in slot {current_api_key_slot} revoked")

old key in slot 1 revoked
